In [ ]:
"""
Specialized Review Agents

This module contains specialized agents for document review:
- GrammarAgent: Checks grammar, punctuation, sentence structure
- FactCheckerAgent: Verifies factual claims and citations
- StyleReviewAgent: Checks tone, clarity, readability

Each agent produces findings in a standardized format.

BUGS TO FIX:
1. Incomplete finding format - missing required fields (severity, confidence)
2. Missing role boundaries in system prompts
3. No error handling for LLM failures
"""

import json
import uuid
from typing import Dict, List, Any, Optional
from abc import ABC, abstractmethod

try:
    from .llm import get_llm_client
except ImportError:
    from llm import get_llm_client


class BaseReviewAgent(ABC):
    """Base class for all review agents"""

    def __init__(self, agent_id: str, agent_type: str):
        self.agent_id = agent_id
        self.agent_type = agent_type
        self.llm = get_llm_client()

    @abstractmethod
    def get_system_prompt(self) -> str:
        """Get the system prompt for this agent"""
        pass

    async def analyze(self, document: Dict[str, Any]) -> Dict[str, Any]:
        content = document.get("content", "")
        title = document.get("title", "Untitled")

        messages = [
            {"role": "system", "content": self.get_system_prompt()},
            {"role": "user", "content": f"Review this document:\n\nTitle: {title}\n\nContent:\n{content}"}
        ]

        try:
            response = self.llm.chat_completion(messages, max_tokens=1000)
            findings = self._parse_findings(response.choices[0].message.content)

            return {"agent": self.agent_id, "agent_type": self.agent_type, "findings": findings}
        except Exception:
            return {"agent": self.agent_id, "agent_type": self.agent_type, "findings": []}

    def _parse_findings(self, response_text: str) -> List[Dict[str, Any]]:
        findings = []
        try:
            parsed = json.loads(response_text)
            if isinstance(parsed, list):
                for item in parsed:
                    finding = {
                        "finding_id": str(uuid.uuid4()),
                        "source_agent": self.agent_id,
                        "category": self.agent_type,
                        "description": item.get("description", item.get("issue", str(item))),
                        "location": item.get("location", "general"),
                        "suggestion": item.get("suggestion", item.get("fix", "")),
                        "severity": item.get("severity", "low"),
                        "confidence": item.get("confidence", 0.0)
                    }
                    findings.append(finding)
            return findings
        except json.JSONDecodeError:
            pass

        lines = response_text.strip().split("\n")
        for line in lines:
            line = line.strip()
            if line and not line.startswith("#"):
                finding = {
                    "finding_id": str(uuid.uuid4()),
                    "source_agent": self.agent_id,
                    "category": self.agent_type,
                    "description": line,
                    "location": "general",
                    "suggestion": "",
                    "severity": "low",
                    "confidence": 0.0
                }
                findings.append(finding)
        return findings


class GrammarAgent(BaseReviewAgent):
    def __init__(self):
        super().__init__("grammar_agent", "grammar")

    def get_system_prompt(self) -> str:
        return """
            You are a grammar reviewer. Check the document for:
                - Grammar mistakes
                - Punctuation errors
                - Sentence structure issues
                - Spelling mistakes

                Return findings as a JSON array with objects containing:
                - description: What the issue is
                - location: Where in the document (paragraph/sentence reference)
                - suggestion: How to fix it

                Do NOT comment on factual accuracy, style, or formatting.
                Suggest improvements for clarity and readability.
                Stay in your lane. Only report on grammar issues.
            """

    def get_default_severity(self) -> str:
        return "medium"


class FactCheckerAgent(BaseReviewAgent):
    def __init__(self):
        super().__init__("fact_checker_agent", "fact_checker")

    def get_system_prompt(self) -> str:
        return """
            You are a fact checker. Review the document for:
                - Factual claims that need verification
                - Missing citations for statistics or data
                - Potentially incorrect information
                - Unsupported assertions

                Return findings as a JSON array with objects containing:
                - description: What the factual issue is
                - location: Where in the document
                - suggestion: How to verify or correct

                Do NOT comment on grammar, style, or formatting.
            """

    def get_default_severity(self) -> str:
        return "high"

class StyleReviewAgent(BaseReviewAgent):
    def __init__(self):
        super().__init__("style_agent", "style")

    def get_system_prompt(self) -> str:
        return """
            You are a style reviewer. Check the document for:
                - Unclear or confusing sentences
                - Inconsistent tone
                - Readability issues
                - Verbose or redundant text

                Return findings as a JSON array with objects containing:
                - description: What the style issue is
                - location: Where in the document
                - suggestion: How to improve

                Do NOT comment on grammar errors or factual accuracy.
            """

    def get_default_severity(self) -> str:
        return "low"


def create_agent(agent_type: str) -> BaseReviewAgent:
    agents = {"grammar": GrammarAgent, "fact_checker": FactCheckerAgent, "style": StyleReviewAgent}
    if agent_type not in agents:
        raise ValueError(f"Unknown agent type: {agent_type}. Valid types: {list(agents.keys())}")
    return agents[agent_type]()


In [ ]:
"""
Review Orchestrator

BUGS TO FIX:
1. Broken team composition - doesn't select agents based on document type
2. Sequential execution instead of parallel
3. No partial failure handling
4. Missing context initialization
"""

import asyncio
from typing import Dict, List, Any, Optional

try:
    from .agents import create_agent, BaseReviewAgent
    from .shared_context import SharedContext, create_shared_context
    from .synthesizer import ReviewSynthesizer
except ImportError:
    from agents import create_agent, BaseReviewAgent
    from shared_context import SharedContext, create_shared_context
    from synthesizer import ReviewSynthesizer


class ReviewOrchestrator:
    def __init__(self):
        self.agents: Dict[str, BaseReviewAgent] = {}
        self.shared_context: Optional[SharedContext] = None
        self.synthesizer = ReviewSynthesizer()
        self._initialize_agents()

    def _initialize_agents(self):
        agent_types = ["grammar", "fact_checker", "style"]
        for agent_type in agent_types:
            self.agents[agent_type] = create_agent(agent_type)

    def compose_team(self, document: Dict[str, Any]) -> List[str]:
        has_factual_claims = self._has_factual_claims(document)
        is_formal_document = self._is_formal_document(document)

        team = []
        if has_factual_claims:
            team.append("fact_checker")
        if is_formal_document:
            team.append("style")

        team.append("grammar")

        return team

    async def run_analysis(self, team: List[str], document: Dict[str, Any]) -> Dict[str, Any]:
        tasks = []
        agent_names = []

        for agent_name in team:
            agent = self.agents.get(agent_name)
            if agent:
                tasks.append(self._safe_analyze(agent_name, document))
                agent_names.append(agent_name)

        gathered_results = await asyncio.gather(*tasks, return_exceptions=True)

        results = {}
        completed = []
        failed = []

        for agent_name, result in zip(agent_names, gathered_results):
            if isinstance(result, Exception) or (isinstance(result, dict) and "error" in result):
                failed.append(agent_name)
            else:
                results[agent_name] = result
                completed.append(agent_name)

        return {"results": results, "completed": completed, "failed": failed}

    async def _safe_analyze(self, agent_name: str, document: Dict[str, Any]) -> Dict[str, Any]:
        try:
            return await self.agents[agent_name].analyze(document)
        except Exception as e:
            return {"error": str(e), "findings": []}

    def _initialize_context(self, document: Dict[str, Any]) -> SharedContext:
        if not self.shared_context:
            self.shared_context = create_shared_context(document)
        return self.shared_context

    async def review_document(self, document: Dict[str, Any]) -> Dict[str, Any]:
        team = self.compose_team(document)
        analysis_results = await self.run_analysis(team, document)
        all_findings = []
        for agent_name, result in analysis_results["results"].items():
            if "findings" in result:
                all_findings.extend(result["findings"])
        review = self.synthesizer.synthesize(all_findings, document)
        # Ensure failed_agents and completed_agents are present at top level
        review["team"] = team
        review["completed_agents"] = analysis_results.get("completed", [])
        review["failed_agents"] = analysis_results.get("failed", [])
        return review

    def _has_factual_claims(self, document: Dict[str, Any]) -> bool:
        if document.get("has_claims"):
            return True
        content = document.get("content", "").lower()
        claim_indicators = ["percent", "%", "million", "billion", "study shows",
                          "research indicates", "according to", "statistics"]
        return any(indicator in content for indicator in claim_indicators)

    def _is_formal_document(self, document: Dict[str, Any]) -> bool:
        formal_types = ["report", "proposal", "article", "whitepaper", "memo"]
        return document.get("document_type", "").lower() in formal_types


In [ ]:
"""
Shared Context Manager - METHODS TO IMPLEMENT
"""
import asyncio
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field
from datetime import datetime


@dataclass
class SharedContext:
    document_id: str
    document_title: str
    document_type: str
    created_at: datetime = field(default_factory=datetime.utcnow)
    _findings: List[Dict[str, Any]] = field(default_factory=list)
    _analyzed_by: Dict[str, List[str]] = field(default_factory=dict)
    _agent_status: Dict[str, str] = field(default_factory=dict)
    _lock: Optional[asyncio.Lock] = field(default=None)

    def __post_init__(self):
        self._lock = asyncio.Lock()

    async def add_finding(self, finding: Dict[str, Any]) -> None:
        self._findings.append(finding)

    async def get_findings(self, agent_type: Optional[str] = None) -> List[Dict[str, Any]]:
        if agent_type:
            return [f for f in self._findings if f.get("agent_type") == agent_type]
        return self._findings

    async def mark_file_analyzed(self, file_path: str, agent_id: str) -> None:
        self._analyzed_by.setdefault(agent_id, []).append(file_path)

    async def get_analysis_status(self) -> Dict[str, Any]:
        return {
            "agents_completed": list(agent_name for agent_name, status in self._agent_status.items() if status == "completed"),
            "agents_failed": list(agent_name for agent_name, status in self._agent_status.items() if status == "failed"),
            "files_analyzed": {
                file_name: agent_name
                for agent_name, files in self._analyzed_by.items()
                for file_name in files
            }
        }

    async def set_agent_status(self, agent_id: str, status: str) -> None:
        self._agent_status[agent_id] = status

    async def acquire_lock(self) -> None:
        await self._lock.acquire()

    async def release_lock(self) -> None:
        self._lock.release()


def create_shared_context(document: Dict[str, Any]) -> SharedContext:
    return SharedContext(
        document_id=document.get("id", "unknown"),
        document_title=document.get("title", "Untitled"),
        document_type=document.get("document_type", "general")
    )


In [ ]:
"""
Review Synthesizer - METHODS TO IMPLEMENT
"""
from typing import Dict, List, Any
from difflib import SequenceMatcher


class ReviewSynthesizer:
    PRIORITY_ORDER = ["grammar", "fact_checker", "style"]
    SEVERITY_LEVELS = ["critical", "high", "medium", "low", "info"]

    def __init__(self, similarity_threshold: float = 0.8):
        self.similarity_threshold = similarity_threshold

    def synthesize(self, findings: List[Dict[str, Any]], document: Dict[str, Any]) -> Dict[str, Any]:
        deduplicated = self._deduplicate_findings(findings)
        resolved = self._resolve_conflicts(deduplicated)
        recommendation = self._determine_recommendation(resolved)
        formatted_review = self._format_review(resolved, recommendation, document)
        summary = f"Review completed with {len(resolved)} findings."
        return {
            "formatted_review": formatted_review,
            "recommendation": recommendation,
            "findings": findings,
            "summary": summary,
        }

    def _deduplicate_findings(self, findings: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        # For each location, keep the finding from the agent with the highest priority
        by_location = {}
        for finding in findings:
            location = finding.get("location", "general")
            agent_type = finding.get("category", finding.get("agent_type", ""))
            if location not in by_location:
                by_location[location] = finding
            else:
                existing = by_location[location]
                existing_type = existing.get("category", existing.get("agent_type", ""))
                if self._get_agent_priority(agent_type) < self._get_agent_priority(existing_type):
                    by_location[location] = finding

        return list(by_location.values())

    def _resolve_conflicts(self, findings: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        # For each (location), keep the finding from the agent with highest priority
        by_location = {}
        for finding in findings:
            location = finding.get("location", "general")
            agent_type = finding.get("category", finding.get("agent_type", ""))
            key = location
            if key not in by_location:
                by_location[key] = finding
            else:
                # Compare priorities
                existing = by_location[key]
                existing_type = existing.get("category", existing.get("agent_type", ""))
                if self._get_agent_priority(agent_type) < self._get_agent_priority(existing_type):
                    by_location[key] = finding
        return list(by_location.values())

    def _determine_recommendation(self, findings: List[Dict[str, Any]]) -> str:
        if not findings:
            return "approve"

        severities = [str(f.get("severity", "low")).lower() for f in findings]
        # If any critical or high, reject
        if any(s in ("critical", "high") for s in severities):
            return "reject"
        # If all low/info, approve
        if all(s in ("low", "info") for s in severities):
            return "approve"
        # Otherwise, revise
        return "revise"

    def _format_review(self, findings: List[Dict[str, Any]], recommendation: str,
                       document: Dict[str, Any]) -> str:
        review_lines = [f"Document Title: {document.get('title', 'Untitled')}",
                        f"Document Type: {document.get('document_type', 'general')}",
                        f"Total Findings: {len(findings)}",
                        "Findings:"]

        for finding in findings:
            review_lines.append(f" - {finding.get('text', '')} (Severity: {finding.get('severity', 'low')})")
        review_lines.append(f"Recommendation: {recommendation}")

        return "\n".join(review_lines)

    def _are_similar(self, text1: str, text2: str) -> bool:
        if not text1 or not text2:
            return False
        ratio = SequenceMatcher(None, text1.lower(), text2.lower()).ratio()
        return ratio >= self.similarity_threshold

    def _get_agent_priority(self, agent_type: str) -> int:
        try:
            return self.PRIORITY_ORDER.index(agent_type)
        except ValueError:
            return len(self.PRIORITY_ORDER)

    def _get_severity_index(self, severity: str) -> int:
        try:
            return self.SEVERITY_LEVELS.index(severity.lower())
        except (ValueError, AttributeError):
            return len(self.SEVERITY_LEVELS)
